<a href="https://colab.research.google.com/github/zo-ff/pandas/blob/main/_Fourier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# 【保存版】DeterministicProcess (DP) と CalendarFourier の仕組みまとめ
# ==============================================================================
# このセルを実行すると、実際にどのような特徴量（列）が生成されるかを目で確認できます。

import pandas as pd
import numpy as np
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess

# ------------------------------------------------------------------------------
# 準備：サンプルの日付インデックス（日次データ）を作成
# ------------------------------------------------------------------------------
# 日次データ（freq='D'）として、2026年6月の1ヶ月分のインデックスを作成します。
# ※DPは、このインデックスの頻度（freq='D'）を自動で読み取ります。
date_index = pd.date_range(start="2026-06-01", end="2026-06-30", freq="D")

# ------------------------------------------------------------------------------
# 1. カレンダーフーリエ（CalendarFourier）の設定
# ------------------------------------------------------------------------------
# 【設定の意味】
# - freq='ME' : 「1ヶ月（月末ベース）」を繰り返しの基準（基本周期）にする。
# - order=4   : 1ヶ月の中に、4つの異なる細かさの波（sin/cos）を作る。
#               第1調和 (1往復) 〜 第4調和 (4往復 ≒ 週次の波) までの計8列が作られます。
fourier_term = CalendarFourier(freq="ME", order=4)

# ------------------------------------------------------------------------------
# 2. DeterministicProcess (DP) の定義
# ------------------------------------------------------------------------------
dp = DeterministicProcess(
    index=date_index,

    # 【const=True】
    # -> すべて 1.0 の「const（定数項）」列を作る。モデルの初期の「ゲタ（切片）」になります。
    constant=True,

    # 【order=1】
    # -> 1, 2, 3... と増える「trend（タイムダミー）」列を作る。長期の直線トレンドを表現します。
    order=1,

    # 【seasonal=True】
    # -> インデックスの 'freq=D' (日次) を自動判定し、「曜日ダミー」を生成します。
    # -> 多重共線性（constとの重複）を防ぐため、1曜日を自動カットした「6列」が作られます。
    # -> これがピリオドグラムで鋭く立っていた Weekly(52) の針を吸収するスイッチになります。
    seasonal=True,

    # 【additional_terms】
    # -> 上で作成したフーリエ項を合流させます（sinが4列、cosが4列の計8列を追加）。
    # -> これが Monthly(12) などの「なだらかな季節の波」を表現します。
    additional_terms=[fourier_term]
)

# ------------------------------------------------------------------------------
# 3. 特徴量テーブル（X）の生成と確認
# ------------------------------------------------------------------------------
# in_sample() で、上記の設定に基づいた具体的な数値のテーブルを生成します。
X = dp.in_sample()

# 生成された列名の一覧を表示
print("【生成された特徴量の列一覧】")
for col in X.columns:
    print(f" - {col}")
print("-" * 80)

# 最初の5行を表示して、実際の数値（スイッチの1.0や、フーリエの波の数値）を確認
print("【生成されたデータのプレビュー（最初の5行）】")
X.head()

【生成された特徴量の列一覧】
 - const
 - trend
 - s(2,7)
 - s(3,7)
 - s(4,7)
 - s(5,7)
 - s(6,7)
 - s(7,7)
 - sin(1,freq=ME)
 - cos(1,freq=ME)
 - sin(2,freq=ME)
 - cos(2,freq=ME)
 - sin(3,freq=ME)
 - cos(3,freq=ME)
 - sin(4,freq=ME)
 - cos(4,freq=ME)
--------------------------------------------------------------------------------
【生成されたデータのプレビュー（最初の5行）】


,const,trend,"s(2,7)","s(3,7)","s(4,7)","s(5,7)","s(6,7)","s(7,7)","sin(1,freq=ME)","cos(1,freq=ME)","sin(2,freq=ME)","cos(2,freq=ME)","sin(3,freq=ME)","cos(3,freq=ME)","sin(4,freq=ME)","cos(4,freq=ME)"
2026-06-01,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
2026-06-02,1.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.207912,0.978148,0.406737,0.913545,0.587785,0.809017,0.743145,0.669131
2026-06-03,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.406737,0.913545,0.743145,0.669131,0.951057,0.309017,0.994522,-0.104528
2026-06-04,1.0,4.0,0.0,0.0,1.0,0.0,0.0,0.0,0.587785,0.809017,0.951057,0.309017,0.951057,-0.309017,0.587785,-0.809017
2026-06-05,1.0,5.0,0.0,0.0,0.0,1.0,0.0,0.0,0.743145,0.669131,0.994522,-0.104528,0.587785,-0.809017,-0.207912,-0.978148
